# Making SIMSOPT GPU native: production-scale profile

Select **Runtime > Change runtime type > GPU** before running the notebook. This workflow benchmarks the 128-by-128 surface, 6-base-coil stress fixture using the full GPU-native engineering objective: quadratic flux, length, coil--coil distance, coil--surface distance, curvature, and mean-squared curvature. The run autotunes 25 custom-VJP tile pairs, measures a seven-sample one-thread CPU baseline, records a Perfetto trace and device-memory profile, evaluates the production gates, and downloads all artifacts.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "gpu-native-objective",
            "https://github.com/PedroFranciscoGil/simsopt.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True
    )
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True
)
os.chdir(repo)
source_root = repo / "src"
source_path = str(source_root)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/gpu"],
    check=True,
)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-production-profile")
if artifact_root.exists():
    shutil.rmtree(artifact_root)

subprocess.run(
    [
        sys.executable,
        "benchmarks/gpu/production_benchmark.py",
        "--problem",
        "stress",
        "--output-dir",
        str(artifact_root),
    ],
    check=True,
)

In [ ]:
import json

summary_file = artifact_root / "stress-summary.json"
summary = json.loads(summary_file.read_text())
print(json.dumps(summary, indent=2))
assert summary["objective_scope"] == "gpu_native_full_engineering"
assert summary["deferred_objective_terms"] == []
assert summary["gates"]["gpu_backend"]["passed"]
assert summary["gates"]["float64_parity"]["passed"]
assert summary["gates"]["steady_state_variation"]["passed"]
assert summary["gates"]["device_memory_fraction"]["passed"]
print("All required correctness, stability, residency, and memory gates passed.")
print("The 3x speed gate is reported, but does not prevent artifact export.")

In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/simsopt-production-profile", "zip", artifact_root
)
files.download(archive)